# Feature Engineering — Dataset Final para Modelagem

## Objetivo

Este notebook transforma os dados brutos de transferências em um dataset pronto para modelagem estatística. 
O objetivo final é testar a hipótese do **prêmio do vendedor**: clubes que realizaram vendas na mesma janela 
pagam mais caro nas compras subsequentes?

Para isolar esse efeito, precisamos:
1. Remover o "preço justo" do jogador (via modelo hedônico) e trabalhar apenas com o **resíduo** (sobrepreço)
2. Controlar por características do clube comprador (tamanho, liga, histórico)
3. Capturar a posição estrutural do clube na rede de transferências

## Pipeline

| Etapa | Descrição |
|-------|----------|
| 1. Filtragem | Apenas compras pagas com fee ≥ €100k e market value válido |
| 2. Winsorização | Clipar outliers extremos (π de +3900%) no percentil 1–99 |
| 3. Log transforms | Corrigir assimetria da distribuição de fees |
| 4. Features de clube | Revenue de vendas, volume, spending na mesma janela |
| 5. Modelo hedônico | OLS para extrair o resíduo (prêmio não explicado pelos fundamentos) |
| 6. Features de rede | Grafo dirigido por temporada → degree, pagerank, strength |
| 7. Exportação | CSV final com ~2100 linhas e ~40 features |

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

df_full = pd.read_csv('../output/transfers_enriched_consolidated.csv')
print(f'Carregado: {df_full.shape}')

Carregado: (16215, 28)


## 1. Filtragem e Limpeza Base

Partimos de 16.215 movimentações brutas (incluindo empréstimos, transferências gratuitas, fim de empréstimo). 
Para a análise de prêmio, precisamos apenas de **compras pagas** (, ) onde tanto 
o fee quanto o market value estão disponíveis. Removemos também transações simbólicas (fee < €100k) que 
representam ruído administrativo.

A **winsorização** a 1–99% é necessária porque a distribuição de  tem caudas extremas 
(até +3900%), que distorceriam qualquer estimador OLS. A transformação logarítmica corrige a forte 
assimetria à direita dos valores monetários (mediana €4.5M vs média €10.3M).

In [2]:
# Filtrar: apenas compras pagas com fee e market_value válidos
df = (
    df_full[
        (df_full['fee_type'] == 'paid') &
        (df_full['direction'] == 'In') &
        (df_full['fee'].notna()) &
        (df_full['market_value'].notna()) &
        (df_full['fee'] >= 100_000)  # remover transações simbólicas
    ]
    .copy()
    .reset_index(drop=True)
)

print(f'Transferências para modelagem: {len(df)}')
print(f'Temporadas: {sorted(df["season_id"].unique())}')
print(f'Clubes compradores únicos: {df["buyer"].nunique()}')

Transferências para modelagem: 2144
Temporadas: [np.int64(2023), np.int64(2024), np.int64(2025)]
Clubes compradores únicos: 159


In [3]:
# Winsorização do premium_ratio a 1-99 percentil
p01 = df['premium_ratio'].quantile(0.01)
p99 = df['premium_ratio'].quantile(0.99)
df['premium_ratio_w'] = df['premium_ratio'].clip(p01, p99)

print(f'premium_ratio original: min={df["premium_ratio"].min():.2f}, max={df["premium_ratio"].max():.2f}')
print(f'premium_ratio_w (1-99%): min={p01:.2f}, max={p99:.2f}')

premium_ratio original: min=-0.99, max=39.00
premium_ratio_w (1-99%): min=-0.88, max=8.43


In [4]:
# Transformações log
df['log_fee'] = np.log1p(df['fee'])
df['log_mv'] = np.log1p(df['market_value'])
df['log_league_mv'] = np.log1p(df['league_total_market_value'])

print('Transformações log aplicadas.')

Transformações log aplicadas.


## 2. Features Derivadas do Clube

Para cada clube comprador em cada temporada, calculamos:

- **revenue_sales**: receita total de vendas na mesma janela — é a variável de tratamento central da hipótese. Se o "prêmio do vendedor" existe, clubes com maior revenue devem pagar mais prêmio.
- **n_sales**: número de vendas realizadas — captura a intensidade da atividade de venda.
- **max_sale**: maior venda individual — vendas "blockbuster" podem ter efeito desproporcional.
- **total_spend** e **n_buys**: volume total de compras — controla o confundidor C1 (clubes ricos compram e vendem mais).
- **net_balance**: receita - gasto — indica se o clube está em modo "investimento" ou "realização de lucro".

Essas features permitem distinguir entre o efeito de "ter dinheiro em caixa" (revenue) e o comportamento habitual de gastos do clube (total_spend).

In [5]:
# Revenue e volume de vendas do clube na mesma temporada
sales = df_full[
    (df_full['fee_type'] == 'paid') &
    (df_full['direction'] == 'Out') &
    (df_full['fee'].notna())
].copy()

club_sales = (
    sales
    .groupby(['club', 'season_id'])
    .agg(
        revenue_sales=('fee', 'sum'),
        n_sales=('fee', 'count'),
        max_sale=('fee', 'max')
    )
    .reset_index()
    .rename(columns={'club': 'buyer'})
)

df = df.merge(club_sales, on=['buyer', 'season_id'], how='left')
df['revenue_sales'] = df['revenue_sales'].fillna(0)
df['n_sales'] = df['n_sales'].fillna(0).astype(int)
df['max_sale'] = df['max_sale'].fillna(0)
df['log_revenue'] = np.log1p(df['revenue_sales'])

print(f'Clubes com vendas na janela: {(df["revenue_sales"] > 0).sum()}/{len(df)} ({(df["revenue_sales"] > 0).mean()*100:.0f}%)')
print(f'Revenue médio (quando >0): €{df.loc[df["revenue_sales"]>0, "revenue_sales"].mean()/1e6:.1f}M')

Clubes com vendas na janela: 2071/2144 (97%)
Revenue médio (quando >0): €64.5M


In [6]:
# Spending do clube na mesma temporada (total de compras)
club_buys = (
    df_full[
        (df_full['fee_type'] == 'paid') &
        (df_full['direction'] == 'In') &
        (df_full['fee'].notna())
    ]
    .groupby(['club', 'season_id'])
    .agg(total_spend=('fee', 'sum'), n_buys=('fee', 'count'))
    .reset_index()
    .rename(columns={'club': 'buyer'})
)

df = df.merge(club_buys, on=['buyer', 'season_id'], how='left')
df['total_spend'] = df['total_spend'].fillna(0)
df['n_buys'] = df['n_buys'].fillna(0).astype(int)

# Net balance: receita - gasto
df['net_balance'] = df['revenue_sales'] - df['total_spend']

print(f'Spend médio por clube/temporada: €{df["total_spend"].mean()/1e6:.1f}M')

Spend médio por clube/temporada: €75.6M


## 3. Modelo Hedônico e Resíduo

O **modelo hedônico** decompõe o preço de uma transferência em duas partes:

$$\log(\text{fee}) = \beta_0 + \beta_1 \log(\text{MV}) + \beta_2 \cdot \text{age} + \beta_3 \cdot \text{position} + \gamma_t + \varepsilon$$

Onde:
- log(MV) captura o "preço justo" baseado na avaliação do Transfermarkt
- age e position controlam características observáveis do atleta
- gamma_t são efeitos fixos de temporada (absorvem inflação de mercado)
- epsilon é o **resíduo** — a parte do preço NÃO explicada pelos fundamentos

O resíduo convertido para percentual (pi_hedonic_pct) é a **variável dependente ideal** para a modelagem causal: 
representa o sobrepreço (ou desconto) que o clube pagou além do que os fundamentos do jogador justificam. 
É sobre esse resíduo que testamos se a receita de vendas tem efeito causal.

In [7]:
# Regressão hedônica: log_fee ~ log_mv + age + position + season
df['is_attacker'] = (df['position_group'] == 'Attacker').astype(int)
df['is_midfielder'] = (df['position_group'] == 'Midfielder').astype(int)
df['is_defender'] = (df['position_group'] == 'Defender').astype(int)

season_dummies = pd.get_dummies(df['season_id'], prefix='season', drop_first=True).astype(int)
df = pd.concat([df, season_dummies], axis=1)

X_cols = ['log_mv', 'age', 'is_attacker', 'is_midfielder', 'is_defender'] + list(season_dummies.columns)
X = sm.add_constant(df[X_cols])
y = df['log_fee']

model = sm.OLS(y, X).fit()
df['log_fee_pred'] = model.predict(X)
df['hedonic_residual'] = y - df['log_fee_pred']

# Converter resíduo para % de prêmio
df['pi_hedonic_pct'] = (np.expm1(df['log_fee']) / np.expm1(df['log_fee_pred']) - 1) * 100

print(f'Modelo Hedônico R² = {model.rsquared:.3f}')
print(f'\nResíduo hedônico (pi_hedonic_pct):')
print(df['pi_hedonic_pct'].describe().round(1))

Modelo Hedônico R² = 0.746

Resíduo hedônico (pi_hedonic_pct):
count    2144.0
mean       30.4
std       125.6
min       -99.1
25%       -32.1
50%         4.6
75%        52.3
max      1633.4
Name: pi_hedonic_pct, dtype: float64


## 4. Features de Rede (por temporada)

O mercado de transferências é modelado como um **grafo dirigido e ponderado**:
- **Nós** = clubes
- **Arestas** = transferências (vendedor para comprador)
- **Peso** = fee pago (euros)

Para cada clube em cada temporada, extraímos:

| Métrica | Significado |
|---------|------------|
| in_degree | Nº de vendedores distintos que forneceram jogadores ao clube |
| out_degree | Nº de compradores distintos que compraram do clube |
| pagerank | Importância estrutural na rede (clubes que compram de clubes importantes têm PR alto) |
| in_strength | Volume total em euros gasto em compras |
| out_strength | Volume total em euros recebido em vendas |
| net_flow | out_strength - in_strength (positivo = exportador líquido) |

Essas métricas capturam a **posição estrutural** do clube no ecossistema, que é um confundidor relevante: 
PSG, Chelsea e Real Madrid dominam o in_strength consistentemente, e essa posição pode explicar parte do prêmio 
independentemente de vendas recentes.

In [8]:
# Construir grafo dirigido por temporada e extrair métricas
paid_all = df_full[
    (df_full['fee_type'] == 'paid') &
    (df_full['direction'] == 'In') &
    (df_full['fee'].notna()) &
    (df_full['buyer'].notna()) &
    (df_full['seller'].notna())
].copy()

network_features = []

for season in sorted(paid_all['season_id'].unique()):
    edges = paid_all[paid_all['season_id'] == season]
    
    G = nx.DiGraph()
    for _, row in edges.iterrows():
        s, b, f = row['seller'], row['buyer'], row['fee']
        if G.has_edge(s, b):
            G[s][b]['weight'] += f
            G[s][b]['count'] += 1
        else:
            G.add_edge(s, b, weight=f, count=1)
    
    pr = nx.pagerank(G, weight='weight')
    
    for node in G.nodes():
        in_str = sum(d['weight'] for _, _, d in G.in_edges(node, data=True))
        out_str = sum(d['weight'] for _, _, d in G.out_edges(node, data=True))
        network_features.append({
            'buyer': node,
            'season_id': season,
            'in_degree': G.in_degree(node),
            'out_degree': G.out_degree(node),
            'pagerank': pr.get(node, 0),
            'in_strength': in_str,
            'out_strength': out_str,
            'net_flow': out_str - in_str,  # positivo = exportador líquido
        })

df_net = pd.DataFrame(network_features)
df = df.merge(df_net, on=['buyer', 'season_id'], how='left')

# Preencher clubes sem presença no grafo
for col in ['in_degree', 'out_degree', 'pagerank', 'in_strength', 'out_strength', 'net_flow']:
    df[col] = df[col].fillna(0)

print(f'Features de rede calculadas para {df_net["buyer"].nunique()} clubes x {df_net["season_id"].nunique()} temporadas')
print(f'Merge com dataset: {df["pagerank"].notna().sum()}/{len(df)} linhas com dados de rede')

Features de rede calculadas para 740 clubes x 3 temporadas
Merge com dataset: 2144/2144 linhas com dados de rede


## 5. Seleção Final e Exportação

Organizamos as features em grupos semânticos para facilitar a modelagem:

- **ID**: identificadores para rastreabilidade (player, buyer, seller, season, league)
- **Target**: 4 variantes da variável dependente (premium_ratio bruto, winsorizado, resíduo log, resíduo %)
- **Player**: características do jogador transferido
- **Club**: features financeiras e estruturais do clube comprador
- **Network**: posição do clube na rede de transferências
- **FE**: dummies de temporada para efeitos fixos

O dataset final está pronto para:
1. **Modelo de painel FE**: 
2. **GNN**: usar o grafo + node features para predizer o prêmio
3. **Análise de subgrupos**: segmentar por liga, tier de clube, faixa etária

In [9]:
# Colunas finais para modelagem
cols_id = ['player', 'buyer', 'seller', 'season_id', 'competition_code', 'country_name']

cols_target = ['premium_ratio', 'premium_ratio_w', 'hedonic_residual', 'pi_hedonic_pct']

cols_player = ['age', 'position_group', 'is_attacker', 'is_midfielder', 'is_defender',
               'log_fee', 'log_mv', 'fee', 'market_value']

cols_club = ['revenue_sales', 'log_revenue', 'n_sales', 'max_sale',
             'total_spend', 'n_buys', 'net_balance',
             'net_transfer_record', 'squad_size', 'average_age',
             'national_team_players', 'log_league_mv']

cols_network = ['in_degree', 'out_degree', 'pagerank', 'in_strength', 'out_strength', 'net_flow']

cols_fe = [c for c in df.columns if c.startswith('season_')]

cols_final = cols_id + cols_target + cols_player + cols_club + cols_network + cols_fe

df_final = df[cols_final].copy()

print(f'Dataset final: {df_final.shape}')
print(f'\nNulos restantes:')
nulls = df_final.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else 'Nenhum!')

Dataset final: (2144, 40)

Nulos restantes:
Nenhum!


In [10]:
# Exportar
df_final.to_csv('../output/transfers_modeling_ready.csv', index=False)
print(f'Exportado: output/transfers_modeling_ready.csv ({df_final.shape[0]} linhas x {df_final.shape[1]} colunas)')

# Resumo
print(f'\n--- RESUMO ---')
print(f'Variáveis dependentes: {cols_target}')
print(f'Features do jogador: {len(cols_player)}')
print(f'Features do clube: {len(cols_club)}')
print(f'Features de rede: {len(cols_network)}')
print(f'Efeitos fixos temporais: {len(cols_fe)}')

Exportado: output/transfers_modeling_ready.csv (2144 linhas x 40 colunas)

--- RESUMO ---
Variáveis dependentes: ['premium_ratio', 'premium_ratio_w', 'hedonic_residual', 'pi_hedonic_pct']
Features do jogador: 9
Features do clube: 12
Features de rede: 6
Efeitos fixos temporais: 3


In [11]:
# Quick sanity check: correlações com a variável dependente
target = 'pi_hedonic_pct'
features = [c for c in cols_player[:-2] + cols_club + cols_network if c != 'position_group']
corr = df_final[features + [target]].corr()[target].drop(target).sort_values(key=abs, ascending=False)
print(f'Correlações com {target}:')
print(corr.round(3).to_string())

Correlações com pi_hedonic_pct:
log_fee                  0.247
log_mv                  -0.178
in_strength              0.151
total_spend              0.151
net_flow                -0.149
national_team_players    0.145
revenue_sales            0.131
log_league_mv            0.128
max_sale                 0.109
pagerank                 0.104
n_buys                   0.080
squad_size              -0.080
in_degree                0.078
net_balance             -0.073
net_transfer_record     -0.060
age                     -0.051
n_sales                  0.037
out_degree              -0.029
is_defender              0.026
is_attacker             -0.025
log_revenue              0.024
out_strength            -0.007
is_midfielder           -0.005
average_age             -0.004
